### Load Files

In [1]:
import pandas as pd 
A = pd.read_csv(r"C:\Users\hp\Downloads\CBAM PROJECT\22-23\ASI_DATA_2022_23_CSV\blkA202223.csv")
B = pd.read_csv(r"C:\Users\hp\Downloads\CBAM PROJECT\22-23\ASI_DATA_2022_23_CSV\blkB202223.csv")
E = pd.read_csv(r"C:\Users\hp\Downloads\CBAM PROJECT\22-23\ASI_DATA_2022_23_CSV\blkE202223.csv")
F = pd.read_csv(r"C:\Users\hp\Downloads\CBAM PROJECT\22-23\ASI_DATA_2022_23_CSV\blkF202223.csv")
G = pd.read_csv(r"C:\Users\hp\Downloads\CBAM PROJECT\22-23\ASI_DATA_2022_23_CSV\blkG202223.csv")
J = pd.read_csv(r"C:\Users\hp\Downloads\CBAM PROJECT\22-23\ASI_DATA_2022_23_CSV\blkJ202223.csv")

### LOAD COLUMN

In [13]:
print(A.columns.tolist())
print(B.columns.tolist())
print(E.columns.tolist())
print(F.columns.tolist())
print(G.columns.tolist())
print(J.columns.tolist())

['yr', 'blk', 'a1', 'a2', 'a3', 'a4', 'a5', 'a7', 'a8', 'a9', 'a10', 'a11', 'a12', 'bonus', 'pf', 'welfare', 'mwdays', 'nwdays', 'wdays', 'costop', 'expshare', 'mult']
['yr', 'blk', 'ab01', 'b02', 'b03', 'b04', 'b05', 'b06f', 'b06t', 'b07', 'b08', 'b09', 'b11']
['yr', 'blk', 'AE01', 'EI1', 'EI3', 'EI4', 'EI5', 'EI6', 'EI7', 'EI8']
['yr', 'blk', 'AF01', 'F1', 'F2A', 'F2B', 'F3', 'F4', 'F5', 'F6', 'F7', 'F8', 'F9', 'F10', 'F11', 'F12', 'F13']
['yr', 'blk', 'AG01', 'G1', 'G2', 'G3', 'G4', 'G5', 'G6', 'G7', 'G8', 'G9', 'G10', 'G11', 'G12']
['yr', 'blk', 'AJ01', 'J11', 'J13', 'J14', 'J15', 'J16', 'J17', 'J18', 'J19', 'J110', 'J111', 'J112', 'J113']


### CBAM NIC Codes AND Filter CBAM Firms

In [17]:
cbam_codes = [
    24101,24102,24103,24104,24105,24106,24107,
    24201,24202,24203,24204,24209,
    23941,23942,
    20121,20122,20129
]
cbam = A[A["a5"].isin(cbam_codes)].copy()

print("CBAM firms:", cbam.shape)

print(cbam["a5"].value_counts())

CBAM firms: (2360, 22)
a5
24105    375
24202    256
24103    254
24209    188
23942    186
24102    148
20129    132
24101    122
23941    115
24104    110
24201    100
24203     95
24106     91
20121     88
20122     61
24204     22
24107     17
Name: count, dtype: int64


### Keep Exporters Only

In [18]:
exporters = cbam[cbam["expshare"] > 0]

print("Exporting CBAM firms:", exporters.shape)

print(exporters["expshare"].describe())

print(
    exporters[["a1","a5","expshare"]]
    .sort_values("expshare", ascending=False)
    .head(20)
)

Exporting CBAM firms: (134, 22)
count    134.000000
mean      26.649254
std       29.194257
min        1.000000
25%        5.000000
50%       14.000000
75%       42.750000
max      100.000000
Name: expshare, dtype: float64
           a1     a5  expshare
34657  140715  24104       100
34648  140706  24104       100
26134  129411  24209       100
44151  155857  24202        99
34662  140720  24104        95
26018  129282  24106        95
30995  135906  24209        94
36737  143612  24106        94
20608  123107  24104        92
37959  145549  20121        90
1962   102098  24103        88
30889  135770  24106        82
34699  140760  24202        80
20684  123193  24202        80
30992  135903  24209        74
30860  135732  24105        71
12152  113821  24209        70
26029  129294  24106        68
34697  140758  24202        65
26007  129268  24105        64


### Employment Aggregation (Block E)

In [19]:
employment = (
    E.groupby("AE01", as_index=False)
     [["EI3","EI5","EI8"]]
     .sum()
)

print(employment.head())

print(employment.shape)

     AE01    EI3    EI5        EI8
0  100002   7572   7572  2484000.0
1  100004   2266   2266   971360.0
2  100005   3930   3930  1970346.0
3  100006      0    610   228000.0
4  100007  14021  14021  5893758.0
(58787, 4)


### Output Aggregation (Block J)

In [20]:
output = (
    J[J["J13"] == 9995000]
    [["AJ01","J113"]]
    .rename(
        columns={
            "AJ01":"factory",
            "J113":"output_value"
        }
    )
)

print(output.head())

print(output.shape)

   factory  output_value
1   100002    17368312.0
3   100004    14026481.0
5   100005    26508402.0
7   100007     1792322.0
9   100008    42727953.0
(48841, 2)


### Merge Employment AND Output

In [22]:
final = exporters.merge(
    employment,
    left_on="a1",
    right_on="AE01",
    how="left"
)

print(final.shape)

print(
    final[["EI3","EI5","EI8"]]
    .isna()
    .sum()
)
final = final.merge(
    output,
    left_on="a1",
    right_on="factory",
    how="left"
)

print(final.shape)

print(
    final[["EI3","EI5","EI8","output_value"]]
    .isna()
    .sum()
)

(134, 26)
EI3    0
EI5    0
EI8    0
dtype: int64
(134, 28)
EI3             0
EI5             0
EI8             0
output_value    0
dtype: int64


### Verify Final Dataset

In [25]:
final = final[
    [
        "a1",
        "a5",
        "a7",
        "costop",
        "expshare",
        "mwdays",
        "wdays",
        "EI3",
        "EI5",
        "EI8",
        "output_value"
    ]
]

final.head()
print("Final dataset:", final.shape)

print(final.describe())

print(final.isna().sum())

Final dataset: (134, 11)
                  a1            a5          a7        costop    expshare  \
count     134.000000    134.000000  134.000000  1.340000e+02  134.000000   
mean   133698.395522  23986.753731   21.820896  2.989634e+10   26.649254   
std     24048.412268    764.998840    8.318700  7.976647e+10   29.194257   
min    102098.000000  20121.000000    3.000000  1.454635e+07    1.000000   
25%    121101.750000  24104.000000   19.000000  1.298263e+09    5.000000   
50%    129281.500000  24105.000000   24.000000  5.731036e+09   14.000000   
75%    140717.250000  24202.000000   27.000000  1.629273e+10   42.750000   
max    214950.000000  24209.000000   36.000000  5.831282e+11  100.000000   

           mwdays       wdays           EI3           EI5           EI8  \
count  134.000000  134.000000  1.340000e+02  1.340000e+02  1.340000e+02   
mean   328.425373  328.641791  2.039644e+06  2.039893e+06  2.647046e+09   
std     28.647817   28.814523  5.898106e+06  5.898037e+06  8.4529

### Save Excel

In [26]:
final.to_excel(
    "CBAM_Exporters_2022_23.xlsx",
    index=False
)

print("Saved successfully")

Saved successfully


### Final Summary

In [30]:
print("CBAM firms:", cbam.shape)
print("Exporting CBAM firms:", exporters.shape)
print("Final dataset:", final.shape)

print(final["expshare"].describe())

CBAM firms: (2360, 22)
Exporting CBAM firms: (134, 22)
Final dataset: (134, 11)
count    134.000000
mean      26.649254
std       29.194257
min        1.000000
25%        5.000000
50%       14.000000
75%       42.750000
max      100.000000
Name: expshare, dtype: float64


### Verify whether you have Block I

In [1]:
import pandas as pd

I = pd.read_csv(r"C:\Users\hp\Downloads\CBAM PROJECT\22-23\ASI_DATA_2022_23_CSV\blkI202223.csv")
print(I.columns.tolist())
print(I.head())

['yr', 'blk', 'AI01', 'II1', 'II3', 'II4', 'II5', 'II6', 'II7']
   yr blk    AI01  II1      II3  II4         II5           II6        II7
0  23   I  100065    1   164000    9  24204719.0  4.695715e+09     194.00
1  23   I  100065    7  9994000    0         0.0  4.695715e+09       0.00
2  23   I  100068    1   137600   27      1010.0  3.395655e+08  336203.42
3  23   I  100068    2   137900   27      1100.0  6.515365e+08  592305.91
4  23   I  100068    7  9994000    0         0.0  9.911020e+08       0.00


In [4]:
print(sorted(I["II3"].unique())[:100])

[np.int64(111100), np.int64(111200), np.int64(112100), np.int64(112200), np.int64(113200), np.int64(115100), np.int64(115200), np.int64(117100), np.int64(117200), np.int64(119000), np.int64(121300), np.int64(121500), np.int64(123100), np.int64(123200), np.int64(123400), np.int64(123500), np.int64(124900), np.int64(125300), np.int64(125900), np.int64(126000), np.int64(129003), np.int64(129099), np.int64(131900), np.int64(134900), np.int64(135300), np.int64(136000), np.int64(137100), np.int64(137200), np.int64(137400), np.int64(137500), np.int64(137600), np.int64(137900), np.int64(139100), np.int64(139900), np.int64(141200), np.int64(144100), np.int64(144400), np.int64(144500), np.int64(144901), np.int64(144999), np.int64(145000), np.int64(149100), np.int64(161001), np.int64(161099), np.int64(162002), np.int64(164000), np.int64(165100), np.int64(165200), np.int64(165300), np.int64(165400), np.int64(165500), np.int64(165600), np.int64(165700), np.int64(165900), np.int64(169000), np.int64(

In [8]:
fuel22 = (
    I.groupby("II3")["II6"]
    .sum()
    .sort_values(ascending=False)
)

fuel22.head(30)

II3
9994000    2.722611e+13
1201000    9.513630e+12
1639001    3.068436e+12
1101002    2.009377e+12
9922100    8.318698e+11
1202003    5.992306e+11
2153500    5.161737e+11
4912999    3.243566e+11
3337000    3.225390e+11
1424004    2.833084e+11
1421000    2.630498e+11
1632001    2.612331e+11
3411071    2.443045e+11
2153100    2.293951e+11
3423200    1.847052e+11
4740300    1.757569e+11
1101006    1.504115e+11
4722199    1.463499e+11
4715002    1.453575e+11
2153300    1.432924e+11
4740100    1.281335e+11
1639006    1.175932e+11
3935001    1.035061e+11
4716000    1.012714e+11
4713002    9.413594e+10
3936102    9.090599e+10
4141201    9.053183e+10
1424099    9.049841e+10
3338007    8.716394e+10
4121101    8.707829e+10
Name: II6, dtype: float64